# Objective :
# - We will push one or multiple PDFs
# - It will go through the ingestion layer
# - Then retrieval and augmentation will happen
# - From all the steps we will see we will use Langchain offered APIs

In [ ]:
!pip install -q langchain langchain-core langchain-openai langchain-community langchain-text-splitters chromadb pypdf tiktoken gradio

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Community Integrations
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma

# Text Splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Langchain Core
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

import gradio as gr

In [ ]:
VECTOR_DB = None

def load_pdfs(files):
  documents = []
  for file in files:
    loader = PyPDFLoader(file.name)
    documents.extend(loader.load())
  return documents

def split_documents(documents):
  splitter = RecursiveCharacterTextSplitter(
      chunk_size = 800, # In my text after every 800 tokens / words a chunk will be created
      chunk_overlap = 150, # For every chunk the first 150 words will overlap with previous chunk
  )
  return splitter.split_documents(documents)

def create_vectorstore(chunks):
  embeddings = OpenAIEmbeddings(model="text-embedding-3-large") #
  vectorstore = Chroma.from_documents(
      documents = chunks,
      embedding = embeddings,
      persist_directory = "./chroma_db"
  )
  return vectorstore

In [ ]:
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer the questions using only the context below .
If the answer is not present in the context , then say:
"I could not find the answer in the provided document(s)"

Context:
{context}

Question:
{question}

Answer very clearly and concisely
""")

def build_rag_chain(vectorstore):
  # Semantic search is happening here ... this line will retrieve the matched chunks but only max 4
  # This is just initialization of the search class ... but it will not execute now
  retriever = vectorstore.as_retriever(search_kwargs={"k":4})
  llm = ChatOpenAI(
      model = "gpt-4o-mini",
      temperature = 0
  )
  rag_chain = (
      {
          "context": retriever,
          "question": RunnablePassthrough(),
      }
      | RAG_PROMPT
      |llm
  )

  return rag_chain

def ingest_pdf(files):
  global VECTOR_DB

  if not files:
    return "upload files"

  documents = load_pdfs(files)
  chunks = split_documents(documents)
  VECTOR_DB = create_vectorstore(chunks)

  return f"Successfully ingested {len(chunks)} chunks from {len(files)} PDFs "

def ask_question(question):
  if VECTOR_DB is None:
    yield "Upload the files and ingest"
    return

  rag_chain = build_rag_chain(VECTOR_DB)

  response = rag_chain.invoke(question) # When I call invoke function here ...the actual execution of the pipeline happens

  # Gradio specific handling
  text = response.content
  partial = ""

  for char in text:
    partial = partial + char
    yield partial


In [ ]:
with gr.Blocks(title = "Langchain based RAG PDF Assistant") as demo:
  gr.Markdown("# Langchain RAG PDF Assistant")
  gr.Markdown(" Upload PDF Documents and ask questions grounded strictly in their content")
  with gr.Row():
    pdf_files = gr.File(
        file_types = [".pdf"],
        file_count = "multiple",
        label = "Upload PDF files"
    )
  ingest_btn = gr.Button(" Ingest PDF")
  ingest_status = gr.Textbox(label = "Ingestion Status")
  ingest_btn.click(
    ingest_pdf,
    inputs = [pdf_files],
    outputs = [ingest_status]
  )

  gr.Markdown("----")

  question = gr.Textbox(
    label = "Ask any question",
    placeholder = "What does the document talk about?"
  )

  ask_btn = gr.Button("Ask")
  answer_box = gr.Markdown(label = "Answer")
  ask_btn.click(
    ask_question,
    inputs = [question],
    outputs = [answer_box]
  )

demo.launch(debug = True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c2c0c4dbf05f683617.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c2c0c4dbf05f683617.gradio.live


In [ ]:
# Llama index a framework for developing LLM Applications ... it can help develop RAG , but it can also help develop Agentic Applications
# llma index is ( langchain + some functionality of langgraph)
# It also supports document loading , ingestion  , search , developing chat agents

# Langchain is able to garner more open source contributions and able to position itself as a single provider of all LLM applications through Langchain + Langgraph ecosystem
# When we only consider llamaindex --- its usecase is limited to RAG and basic agent

In [ ]:
FAISS , ChromaDB ( in memory db) --- are not used

In [ ]:
pinecone, weaviate , cohere , aws , azure--

In [ ]:
# You can write APIs on vector DB ( to ingest , search , update , delete entries)
# You can fall back on Langchain / llamaindex to do that job..they basically create wrapper
# Query --- no SQL query
# Query -- Semantic Query ( Vector needs to be the input )